# Análisis Exploratorio de Datos — Enfoque ROI

**Objetivo:** Explorar el portafolio de automatizaciones RPA desde la perspectiva del retorno de inversión (ROI), identificando qué procesos generan más valor, cuáles factores determinan el ROI y qué características debe tener un buen candidato a automatización.

**Fórmula de ROI utilizada:**
$$ROI\% = \frac{T_{manual} \times ValorHora \times N_{ejec} - T_{robot} \times C_{robot} \times N_{ejec}}{T_{robot} \times C_{robot} \times N_{ejec}} \times 100$$

Donde $C_{robot} = 7.300$ COP/hora es el costo operativo estándar del robot, calculado como:

- Servidor Azure: **150.000 COP/mes**
- Licencia UiPath robot + orquestador: **5.200.000 COP/mes**
- Total: **5.350.000 COP/mes ÷ ~730 h/mes ≈ 7.300 COP/h**

Este valor se aplica de forma uniforme a todas las soluciones del portafolio para que los ROIs sean comparables entre proyectos.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.roi_calculator import build_roi_dataset, get_roi_summary, COSTO_HORA_ROBOT_COP

pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DB_PATH = '../data/database/Procesos_clean.db'
print('Librerías cargadas correctamente.')

## 1. Carga y exploración inicial de datos

In [ ]:
conn = sqlite3.connect(DB_PATH)

df_exec  = pd.read_sql('SELECT * FROM RegistrosDPA_clean', conn)
df_manual = pd.read_sql('SELECT * FROM TiemposManuales_clean', conn)
df_roles  = pd.read_sql('SELECT * FROM RolesAreas_clean', conn)

conn.close()

print(f'RegistrosDPA_clean:    {len(df_exec):>8,} filas  |  {df_exec.shape[1]} columnas')
print(f'TiemposManuales_clean: {len(df_manual):>8,} filas  |  {df_manual.shape[1]} columnas')
print(f'RolesAreas_clean:      {len(df_roles):>8,} filas  |  {df_roles.shape[1]} columnas')

In [ ]:
# Calidad de datos
for name, df in [('RegistrosDPA_clean', df_exec), ('TiemposManuales_clean', df_manual), ('RolesAreas_clean', df_roles)]:
    print(f'\n=== {name} ===')
    null_pct = (df.isnull().sum() / len(df) * 100).round(1)
    print(null_pct[null_pct > 0].to_string())
    if null_pct.sum() == 0:
        print('Sin valores nulos.')

## 2. Cálculo del ROI por automatización

In [ ]:
df_roi = build_roi_dataset()
summary = get_roi_summary(df_roi)

print('=== Resumen ejecutivo del portafolio RPA ===')
print(f"  Total de bots analizados:  {summary['total_bots']}")
print(f"  Bots con ROI calculable:   {summary['bots_con_roi']}")
print(f"  Ahorro neto total:         ${summary['ahorro_total_cop']/1e6:,.1f}M COP")
print(f"  ROI promedio:              {summary['roi_promedio_pct']:.0f}%")
print(f"  ROI mediano:               {summary['roi_mediano_pct']:.0f}%")
print(f"  Tiempo ahorrado total:     {summary['tiempo_ahorrado_horas']:,.0f} horas")
print(f"  Mejor bot por ROI:         {summary['mejor_bot']}")

In [ ]:
# Vista general del dataset de ROI
cols_show = ['Automatizacion','Tecnologia','Estado','Num_Ejecuciones',
             'TiempoManualHoras','DuracionPromedio_Horas','ValorHoraPromedio',
             'Beneficio_Bruto_COP','Costo_Robot_COP','Ahorro_Neto_COP','ROI_Porcentaje']
df_roi[[c for c in cols_show if c in df_roi.columns]].sort_values('ROI_Porcentaje', ascending=False).head(10)

## 3. Distribución del ROI

In [ ]:
df_valid = df_roi.dropna(subset=['ROI_Porcentaje'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma ROI
axes[0].hist(df_valid['ROI_Porcentaje'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_valid['ROI_Porcentaje'].median(), color='red', linestyle='--', label=f"Mediana: {df_valid['ROI_Porcentaje'].median():.0f}%")
axes[0].axvline(df_valid['ROI_Porcentaje'].mean(), color='orange', linestyle='--', label=f"Media: {df_valid['ROI_Porcentaje'].mean():.0f}%")
axes[0].set_title('Distribución del ROI (%)')
axes[0].set_xlabel('ROI (%)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot por tecnología
tecnologias = df_valid.dropna(subset=['Tecnologia'])
order = tecnologias.groupby('Tecnologia')['ROI_Porcentaje'].median().sort_values(ascending=False).index
sns.boxplot(data=tecnologias, x='Tecnologia', y='ROI_Porcentaje', order=order, ax=axes[1], palette='Set2')
axes[1].set_title('ROI por Tecnología')
axes[1].set_xlabel('')
axes[1].set_ylabel('ROI (%)')

plt.tight_layout()
plt.savefig('../reports/figures/roi_distribucion.png', bbox_inches='tight')
plt.show()

## 4. Top 10 automatizaciones por ROI y por ahorro neto

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Top 10 por ROI (%)', 'Top 10 por Ahorro Neto (M COP)'])

top_roi = df_valid.nlargest(10, 'ROI_Porcentaje')
fig.add_trace(go.Bar(
    x=top_roi['ROI_Porcentaje'], y=top_roi['Automatizacion'],
    orientation='h', marker_color='steelblue', name='ROI %'
), row=1, col=1)

top_ahorro = df_valid.nlargest(10, 'Ahorro_Neto_COP').copy()
top_ahorro['Ahorro_M'] = top_ahorro['Ahorro_Neto_COP'] / 1e6
fig.add_trace(go.Bar(
    x=top_ahorro['Ahorro_M'], y=top_ahorro['Automatizacion'],
    orientation='h', marker_color='seagreen', name='Ahorro M COP'
), row=1, col=2)

fig.update_layout(height=500, showlegend=False, title_text='Ranking de automatizaciones')
fig.update_xaxes(title_text='ROI (%)', row=1, col=1)
fig.update_xaxes(title_text='Ahorro Neto (M COP)', row=1, col=2)
fig.show()

## 5. ROI por área y tecnología

In [ ]:
# Unir con área del proceso
area_por_bot = df_exec.groupby('Automatizacion')['Area'].agg(lambda x: x.mode()[0] if len(x) > 0 else 'Desconocido').reset_index()
df_roi_area = df_valid.merge(area_por_bot, on='Automatizacion', how='left')

fig, ax = plt.subplots(figsize=(14, 5))
area_roi = df_roi_area.groupby('Area')['ROI_Porcentaje'].agg(['mean','count']).reset_index()
area_roi = area_roi[area_roi['count'] >= 2].sort_values('mean', ascending=True)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in area_roi['mean']]
ax.barh(area_roi['Area'], area_roi['mean'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('ROI promedio por Área Organizacional')
ax.set_xlabel('ROI promedio (%)')
for i, (v, n) in enumerate(zip(area_roi['mean'], area_roi['count'])):
    ax.text(v + 5, i, f'n={n}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/roi_por_area.png', bbox_inches='tight')
plt.show()

## 6. Tendencia temporal de ahorros acumulados

In [ ]:
# Ejecuciones con bots que tienen ROI calculado
bots_con_roi = df_valid.set_index('Automatizacion')
df_exec['Fecha'] = pd.to_datetime(df_exec['Fecha_ejecucion'], errors='coerce')

df_temporal = df_exec[df_exec['Automatizacion'].isin(bots_con_roi.index)].copy()
df_temporal = df_temporal.merge(
    bots_con_roi[['TiempoManualHoras','DuracionPromedio_Horas','ValorHoraPromedio']].reset_index(),
    on='Automatizacion', how='left'
)
# Fallback: si no hay duración del robot, asumir 10% del tiempo manual
df_temporal['DuracionPromedio_Horas'] = df_temporal['DuracionPromedio_Horas'].fillna(
    df_temporal['TiempoManualHoras'] * 0.1
)
df_temporal['Ahorro_Ejecucion'] = (
    df_temporal['TiempoManualHoras'] * df_temporal['ValorHoraPromedio']
    - df_temporal['DuracionPromedio_Horas'] * COSTO_HORA_ROBOT_COP
)
df_temporal['YearMonth'] = df_temporal['Fecha'].dt.to_period('M')

ahorro_mensual = df_temporal.groupby('YearMonth')['Ahorro_Ejecucion'].sum().reset_index()
ahorro_mensual['YearMonth_str'] = ahorro_mensual['YearMonth'].astype(str)
ahorro_mensual['Ahorro_M_COP'] = ahorro_mensual['Ahorro_Ejecucion'] / 1e6
ahorro_mensual['Acumulado_M_COP'] = ahorro_mensual['Ahorro_M_COP'].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(ahorro_mensual['YearMonth_str'], ahorro_mensual['Ahorro_M_COP'],
            color='steelblue', alpha=0.8, width=0.8)
axes[0].set_title('Ahorro mensual del portafolio RPA')
axes[0].set_ylabel('Ahorro (M COP)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].fill_between(ahorro_mensual['YearMonth_str'],
                     ahorro_mensual['Acumulado_M_COP'], alpha=0.3, color='green')
axes[1].plot(ahorro_mensual['YearMonth_str'],
             ahorro_mensual['Acumulado_M_COP'], color='green', linewidth=2)
axes[1].set_title('Ahorro acumulado del portafolio RPA')
axes[1].set_ylabel('Ahorro acumulado (M COP)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/ahorro_temporal.png', bbox_inches='tight')
plt.show()

## 7. Análisis de correlaciones — factores que determinan el ROI

In [ ]:
num_cols = ['TiempoManualHoras','DuracionPromedio_Horas','ValorHoraPromedio',
            'Num_Ejecuciones','PromTransacciones','TasaExito','TasaError',
            'EjecucionesPorDia','DiasEnProduccion','ROI_Porcentaje','Ahorro_Neto_COP']

df_corr = df_valid[[c for c in num_cols if c in df_valid.columns]].copy()
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Correlación entre variables — dataset ROI', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/correlaciones_roi.png', bbox_inches='tight')
plt.show()

# Correlación con ROI_Porcentaje
print('\nCorrelación con ROI_Porcentaje (|r| mayor = más relevante para predicción):')
print(corr_matrix['ROI_Porcentaje'].drop('ROI_Porcentaje').abs().sort_values(ascending=False).to_string())

## 8. Segmentación: cuadrantes de valor

In [ ]:
# Cuadrante: Alto ROI / Alto Volumen → estrella; Bajo ROI / Bajo Volumen → revisar
med_roi = df_valid['ROI_Porcentaje'].median()
med_exec = df_valid['Num_Ejecuciones'].median()

def cuadrante(row):
    alto_roi = row['ROI_Porcentaje'] >= med_roi
    alto_vol = row['Num_Ejecuciones'] >= med_exec
    if alto_roi and alto_vol: return 'Estrella (alto ROI + alto volumen)'
    if alto_roi and not alto_vol: return 'Nicho (alto ROI + bajo volumen)'
    if not alto_roi and alto_vol: return 'Escalar (bajo ROI + alto volumen)'
    return 'Revisar (bajo ROI + bajo volumen)'

df_valid['Cuadrante'] = df_valid.apply(cuadrante, axis=1)

fig = px.scatter(
    df_valid, x='Num_Ejecuciones', y='ROI_Porcentaje',
    color='Cuadrante', size='Ahorro_Neto_COP', hover_name='Automatizacion',
    color_discrete_map={
        'Estrella (alto ROI + alto volumen)': '#2ecc71',
        'Nicho (alto ROI + bajo volumen)': '#3498db',
        'Escalar (bajo ROI + alto volumen)': '#f39c12',
        'Revisar (bajo ROI + bajo volumen)': '#e74c3c',
    },
    log_x=True, title='Cuadrantes de valor del portafolio RPA',
    labels={'Num_Ejecuciones': 'Número de ejecuciones (log)', 'ROI_Porcentaje': 'ROI (%)'}
)
fig.add_hline(y=med_roi, line_dash='dash', line_color='gray')
fig.add_vline(x=med_exec, line_dash='dash', line_color='gray')
fig.update_layout(height=500)
fig.show()

print('\nDistribución por cuadrante:')
print(df_valid['Cuadrante'].value_counts().to_string())

## 9. Análisis de madurez — ROI vs tiempo en producción

In [ ]:
df_madurez = df_valid.dropna(subset=['DiasEnProduccion', 'ROI_Porcentaje'])

fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(df_madurez['DiasEnProduccion'],
                     df_madurez['ROI_Porcentaje'],
                     c=df_madurez['Ahorro_Neto_COP'], cmap='YlGn',
                     s=80, alpha=0.7, edgecolors='white')
plt.colorbar(scatter, label='Ahorro Neto (COP)')

# Tendencia
z = np.polyfit(df_madurez['DiasEnProduccion'], df_madurez['ROI_Porcentaje'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_madurez['DiasEnProduccion'].min(), df_madurez['DiasEnProduccion'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.7, label=f'Tendencia lineal')

ax.set_xlabel('Días en producción')
ax.set_ylabel('ROI (%)')
ax.set_title('Madurez del bot vs ROI')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/madurez_vs_roi.png', bbox_inches='tight')
plt.show()

## 10. Conclusiones y hallazgos para el modelo predictivo

In [ ]:
print('=== HALLAZGOS CLAVE ===')
print()

pct_positivo = (df_valid['ROI_Porcentaje'] > 0).mean() * 100
print(f'1. {pct_positivo:.0f}% de las automatizaciones con datos tienen ROI positivo.')

mejor = df_valid.loc[df_valid['ROI_Porcentaje'].idxmax()]
print(f'2. Mejor ROI: {mejor["Automatizacion"]} con {mejor["ROI_Porcentaje"]:.0f}%')

mayor_ahorro = df_valid.loc[df_valid['Ahorro_Neto_COP'].idxmax()]
print(f'3. Mayor ahorro: {mayor_ahorro["Automatizacion"]} (${mayor_ahorro["Ahorro_Neto_COP"]/1e6:.1f}M COP)')

corr_manual = abs(df_valid['TiempoManualHoras'].corr(df_valid['ROI_Porcentaje']))
print(f'4. Correlación TiempoManual → ROI: {corr_manual:.2f} (mayor tiempo manual = mayor ROI potencial)')

print()
print('=== FEATURES RECOMENDADOS PARA EL MODELO PREDICTIVO ===')
features_model = [
    'TiempoManualHoras    → mayor predictor de ROI',
    'ValorHoraPromedio    → multiplica directamente el beneficio',
    'Num_Ejecuciones      → escala el ahorro total',
    'DuracionPromedio_Horas → costo del robot',
    'EjecucionesPorDia    → intensidad de uso',
    'TasaExito            → calidad de la automatización',
    'Tecnologia           → plataforma RPA usada',
    'DiasEnProduccion     → madurez del proceso',
]
for f in features_model:
    print(f'  • {f}')

In [ ]:
# Guardar dataset de ROI para el notebook del modelo
import os
os.makedirs('../reports/figures', exist_ok=True)
df_roi.to_csv('../data/roi_dataset.csv', index=False)
print(f'Dataset de ROI guardado: {len(df_roi)} filas → data/roi_dataset.csv')